In [1]:
import duckdb

con = duckdb.connect("C:\dbt-labs\ecommerce\dev.duckdb")

print(con.sql("SHOW TABLES"))

##print(con.sql("SELECT * FROM customers"))

<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\eugsa\AppData\Local\Temp\ipykernel_31008\3308625718.py:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  con = duckdb.connect("C:\dbt-labs\ecommerce\dev.duckdb")


┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ customers           │
│ my_first_dbt_model  │
│ my_second_dbt_model │
└─────────────────────┘



In [2]:
print(con.sql("SELECT schema_name, view_name, sql FROM duckdb_views() where schema_name = 'main';"))

┌─────────────┬──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ schema_name │      view_name       │                                                                                                                                                                                                   sql                                                                                                                                                                                                    │
│   varchar   │       varchar        │                                                                                              

In [3]:
print(con.sql("SELECT * FROM customers"))

┌─────────────┬───────────────┬────────────┐
│ customer_id │ customer_name │  country   │
│    int32    │    varchar    │  varchar   │
├─────────────┼───────────────┼────────────┤
│           1 │ Eugenio       │ Costa Rica │
│           2 │ Maria         │ Costa Rica │
│           3 │ Carlos        │ Panama     │
│           4 │ Ana           │ Mexico     │
└─────────────┴───────────────┴────────────┘



In [4]:
df = con.sql("SELECT * FROM customers").df()

In [5]:
df

,customer_id,customer_name,country
0,1,Eugenio,Costa Rica
1,2,Maria,Costa Rica
2,3,Carlos,Panama
3,4,Ana,Mexico


In [6]:
type(df)


pandas.core.frame.DataFrame

In [7]:
import pandas as pd


type(pd)

module

In [8]:


df.head()

,customer_id,customer_name,country
0,1,Eugenio,Costa Rica
1,2,Maria,Costa Rica
2,3,Carlos,Panama
3,4,Ana,Mexico


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    4 non-null      int32 
 1   customer_name  4 non-null      object
 2   country        4 non-null      object
dtypes: int32(1), object(2)
memory usage: 212.0+ bytes


In [15]:
sum(df.isnull().sum())

0

In [11]:
df["customer_id"].is_unique

True

In [28]:
def check_data_quality(df):

    if sum(df.isnull().sum()) == 0 and df["customer_id"].is_unique:
        return True

    else:
        return False

In [42]:
check_data_quality(df)

Data quality: PASS


In [37]:
df_bad = df.copy()

df_bad.loc[len(df_bad)] = [4, "Ana Duplicate", "Mexico"]

In [38]:
if check_data_quality(df_bad):
    print("Continue pipeline")
else:
    print("Stop pipeline")

FAIL: Duplicate customer_id found
Stop pipeline


In [48]:
def check_data_quality(df):

    nulls = df.isnull().sum()
    unique_ids = df["customer_id"].is_unique

    if nulls.sum() > 0:
        print("FAIL: Null values found")
        print(nulls[nulls > 0])

    if not unique_ids:
        print("FAIL: Duplicate customer_id found")

    if nulls.sum() == 0 and unique_ids:
        return True
    else:
        return False

In [49]:
df_bad_name = df.copy()

df_bad_name.loc[1, "customer_name"] = None

In [52]:
result = check_data_quality(df)
print(result)
result = check_data_quality(df_bad_name)
print(result)


True
FAIL: Null values found
customer_name    1
dtype: int64
False
